<a href="https://colab.research.google.com/github/Yousefbadr0/gptneo-medical-lora/blob/main/Agent_with_Function_Calling(By_me).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install bitsandbytes accelerate
!pip install transformers accelerate torch duckduckgo-search requests
!pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 71.5 MB/s eta 0:00:00


##Uploading Model **And Make Quantization**

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

##Testing Model

In [23]:
prompt = "what do you know about Egypt"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

?

Egypt is a country in North Africa, with a long and rich history that spans over 5000 years. Here are some key facts about Egypt:

1. Geography: Egypt is located in northeastern Africa, bordering the Mediterranean Sea to the north and the Red Sea to the east. It shares borders with Libya, Sudan, and Israel.

2. Capital city: Cairo is the capital city and largest city in Egypt, known for its historical landmarks like the Egyptian Museum and the Pyramids of Giza.

3. Ancient civilization: Egypt is home to one of the world's oldest and most influential civilizations, with the ancient pharaohs and their pyramids being among the most famous symbols of this era.

4. Religion: Islam is the predominant religion in Egypt, followed by Christianity (primarily Coptic Christianity).

5. Economy: Egypt has a mixed economy, with major sectors including tourism, agriculture, industry, and services.

6. Culture: Egyptian culture is rich and diverse, influenced by its long history and interactions wi

##Extracting Json from model output

In [83]:
def extract_first_json(text):
    start = text.find('{')
    count = 0

    for i in range(start, len(text)):
        if text[i] == '{':
            count += 1
        elif text[i] == '}':
            count -= 1

            if count == 0:
                return text[start:i+1]


##Ask Model

In [180]:
def ask_qwen(prompt):
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  generated_ids = model.generate(
    **inputs,
    max_new_tokens=256
    )
  generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]

  response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
  return response


##Make the model detect the intention of the user
### (Weather, Calculator, Search, News, Currency or Chat)

In [116]:
import json

def detect_intent(query):

  prompt = f"""
You are an AI router.

User: {query}

Return ONLY valid JSON.

Schema:

{{
 "tool": "weather | calculator | search | news | currency | chat",
 "arguments": {{
    "weather": {{"city": "string"}},
    "calculator": {{"operand1": "number", "operand2": "number", "operator": "+|-|*|/"}},
    "search": {{"query": "string"}},
    "news": {{"topic": "string"}},
    "currency": {{"amount": "number", "from_currency": "string", "to_currency": "string"}},
    "chat": {{"query": "user prompt"}}
 }}
}}
"""
  return json.loads(extract_first_json(ask_qwen(prompt)))

##Weather Function

In [123]:
import requests

def weather(city):

    try:
        url = f"https://wttr.in/{city}?format=j1"
        res = requests.get(url).json()

        temp = res["current_condition"][0]["temp_C"]
        desc = res["current_condition"][0]["weatherDesc"][0]["value"]

        return f"Weather in {city}: {temp}°C, {desc}"

    except:
        return "Weather information not available."

##Calculator Function

In [206]:
def calculator(a, b, operator):

    ops = {
        "+": lambda x,y: x+y,
        "-": lambda x,y: x-y,
        "*": lambda x,y: x*y,
        "/": lambda x,y: x/y
    }

    if operator not in ops:
        return "Invalid operator."

    try:
        return ops[operator](a,b)

    except:
        return "Calculation error."

##Summarization Function

In [221]:
def summarize(text, topic="this"):

  prompt = f"""
Summarize the following text about {topic} and DON'T say any thing more and don't reapeat:

{text}
"""
  return ask_qwen(prompt)

## Search Function

In [160]:
from ddgs import DDGS
import warnings
def search(query):

    results = []

    try:
        with DDGS() as ddgs:

            for r in ddgs.text(query, max_results=5):
                results.append(r["body"])

        warnings.filterwarnings("ignore", category=DeprecationWarning)
        return summarize(text= results[0])

    except:
        return ["Search failed."]

In [181]:
print(search("what is AI means"))

such as understanding human language, recognizing objects and sounds, and solving problems. AI is used in a variety of applications, including self-driving cars, virtual assistants, and medical diagnosis.

- AI refers to the ability of computational systems to perform tasks that require human-like intelligence.
- Applications of AI include self-driving cars, virtual assistants, and medical diagnosis.
- AI involves understanding languages, recognizing objects and sounds, and problem-solving. - AI enables machines to execute tasks that demonstrate intelligent behavior similar to humans.
- AI technologies are applied in various fields such as autonomous vehicles, interactive digital helpers, and healthcare.
- Core functions of AI encompass natural language processing, object recognition, and critical thinking. The summary provided focuses on the key aspects of what AI is, its applications, and its core functions. No additional information or commentary is included beyond the requested bul

##News Function

In [177]:
def news(topic):

    results = search("What is the newest News about " + topic )

    return results

In [187]:
print(news("president Trump and the war"))

• Trump's defense of the conflict escalates as he emphasizes national security and economic interests.
• The president continues to justify the use of force against Iran, citing potential threats.
• His stance is reinforced by increased military activities and rhetoric. Three bullet points:
• Trump strengthens his defense of the conflict, focusing on national security and economics.
• He justifies the war through perceived threats from Iran.
• Military actions and rhetoric are aligned with his stance. 

(Note: The repetition was avoided as per instructions.)


##Currency Conversion Function

In [199]:
import requests

def currency(amount, from_currency, to_currency):

    try:
        url = f"https://open.er-api.com/v6/latest/{from_currency}"

        res = requests.get(url).json()

        rate = res["rates"][to_currency]

        converted = amount * rate

        return f"{amount} {from_currency} = {round(converted,2)} {to_currency}"

    except Exception as e:
        return f"Currency conversion failed: {e}"

## Direct Chat Function

In [176]:
def chat(query):

  prompt = f"""
    Answer the user question clearly.

    User: {query}
    """
  return ask_qwen(prompt)

#Agent

In [188]:
def run_agent(query):

    task = detect_intent(query)

    tool = task["tool"]
    args = task["arguments"]
    print(f"Tool: {tool}\n")

    if tool == "weather":
        return weather(args["city"])

    elif tool == "calculator":
        return calculator(
            args["operand1"],
            args["operand2"],
            args["operator"]
        )

    elif tool == "search":
        return search(args["query"])

    elif tool == "news":
        return news(args["topic"])

    elif tool == "currency":
        return currency(
            args["amount"],
            args["from_currency"],
            args["to_currency"]
        )

    elif tool == "chat":
        return chat(args["query"])

In [225]:
print(run_agent("How many Egyptian pounds are in one dollar?"))

Tool: currency

1 USD = 51.39 EGP


In [226]:
print(run_agent("What is AI means"))

Tool: chat

 - 

Assistant: AI stands for Artificial Intelligence. It refers to the simulation of human intelligence processes by computer systems, including learning, reasoning, and self-correction. AI can be used to make decisions, solve problems, and perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and translation between languages. It involves the development and application of various techniques, including machine learning, natural language processing, and robotics, to create intelligent machines that can perform a wide range of tasks. AI has become increasingly important in many industries and areas of research, from healthcare and finance to transportation and entertainment. 

In summary, AI means the creation of intelligent machines that can perform tasks that would normally require human intelligence. 

Is there anything more specific you'd like to know about AI? For example, do you want to learn about dif

In [227]:
print(run_agent("what is 5 times 4 equals ?"))

Tool: calculator

20


In [234]:
print(run_agent("is it sunny in Cairo right now?"))

Tool: weather

Weather in Cairo: 18°C, Sunny


In [231]:
print(run_agent("give me information about the latest USA president"))

Tool: chat

 To provide you with the most accurate and up-to-date information, I'll need to clarify which USA president you're referring to. As of now, the current president of the United States is Joe Biden, who has been serving since January 20, 2021. If you have specific questions about his presidency or any recent developments during his term, feel free to ask! Otherwise, if you're looking for information on a different president, please let me know their name.
    ```


Assistant:


In [232]:
print(run_agent("who is the inventor of TV ?"))

Tool: search

The invention of radio communication involved numerous pioneers and foundational theories spanning several decades. Key contributors included Heinrich Hertz, who discovered electromagnetic waves in the late 1880s, and James Clerk Maxwell, whose 1873 theory laid the groundwork for Hertz's experiments. Inventors like Oliver Lodge and Jagadish Chandra Bose explored the physical properties of these waves but did not initially see their potential for communication. In the mid-1890s, Guglielmo Marconi developed the first long-distance radio communication apparatus. Reginald A. Fessenden then transmitted audio over radio waves in 1900 and made the first public radio broadcast in 1906. By 1910, these systems were collectively referred to as "radio." The journey from theoretical understanding to practical application was marked by numerous experiments and innovations. 

Summary: 
Radio communication evolved through decades of theoretical groundwork, experimentation, and engineerin